# miniG 模型试用

https://huggingface.co/CausalLM/miniG


该模型是在包含超过1.2 亿个条目的综合数据集上进行训练的，该数据集是通过利用大型上下文窗口的最先进的语言模型以及类似于检索增强生成和知识图集成的方法生成的，其中数据合成是在源自 200 亿个令牌的预训练语料库的集群内进行的，随后由模型本身执行验证。

尽管没有完全符合人类偏好，但该模型没有义务迎合结构不良的提示或传统基准中常见的陈词滥调。奖励：其中包括经过锁定图像调整的视觉语言模型的实现。

**支持的输入方式**：文本、图像。对于纯文本权重，请使用https://huggingface.co/CausalLM/miniG/tree/text-only上的分支revision=text-only 。合并 PR #9194后，纯文本的GGUF应该可以工作。

**上下文窗口**： 1M Token

**模型参数**： LLM -9B（从THUDM/glm-4-9b-chat-1m初始化）；可选 ViT-5B

**注意事项**：强烈建议使用标准化实现进行推理，例如 Hugging Face Transformers，以避免使用 vllm 或 lmdeploy 等加速内核时可能发生的性能显着下降 - 更不用说模型量化的潜在灾难性影响。截至目前，已知这些加速推理实现严重损害了有效的视觉推理，尽管它们对纯文本性能的影响不太明显

**推理参数**：我们的观察表明，如果希望获得较少幻觉的结果，建议采用 top_p=0​​.8 采样，然后温度设置为 0.3，或者使用设置为 0.2 的纯温度采样。一般来说，与类似模型相比，需要更低的温度，我们暂时将其归因于对庞大数据集的过度拟合。模型推断应参考THUDM/glm-4-9b-chat-1m和THUDM/glm-4v-9b。我们仅在使用变压器进行推理时保证最佳性能。在我们的测试中，我们还使用了 lmdeploy，这导致多模式输入的性能显着下降。

**关于格式**：我们强烈建议您仔细检查您的输入，以确保： 1. 系统提示不为空。即使是像“你是一个有用的助手”这样简单的事情。预计。 2. <|role|> 标记后始终有一个换行符。这将有助于确保正确解析和处理您的输入。

**关于基准分数**：一般来说，你不应该太担心它们，因为人们总是可以进行专门的训练来取得好的成绩。我们主要将它们用作冒烟测试，快速检查以确保没有发生重大回归。事实上，如果你真正通读基准问题本身，你经常会发现自己嘲笑它们是多么的愚蠢、低质量，甚至是彻头彻尾的愚蠢

**关于训练**：最终发布的版本是使用多个候选模型的合并进行训练的，以试图提高性能。然而，我们无法最终确定这是否有效。排除候选版本，一天之内应该可以在 8*A100-80G 的 16 个节点上实现高效的简单微调。据此，我们估计碳排放量为 700 千克二氧化碳当量。

In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'hf-mirror.com'  # 使用镜像站点
os.environ['HF_HOME'] = '/root/autodl-tmp/cache'  # 设置缓存目录
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '300'  # 设置超时时间为300秒
os.environ['HF_HUB_DOWNLOAD_MAX_RETRIES'] = '5'  # 设置最大重试次数为5

In [ ]:
# 下载模型，这里请注意分支，我就没只需要下载text-only分支的模型
from huggingface_hub import snapshot_download

model_name = "CausalLM/miniG"
branch_name = "text-only"

download_model_path = snapshot_download(
    repo_id=model_name, 
    revision=branch_name
    )

In [ ]:
# 加载 tokenizer 和 text-generation model, 显存占用实测在 16G 左右
import torch 
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(download_model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    download_model_path, 
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    device_map="auto",
    ).eval()

In [ ]:
# 定义对话函数
def chat(query):
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": query}],
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    )
    
    inputs = inputs.to(model.device)
    gen_kwargs = {"max_length": 2500, "do_sample": True, "top_k": 1}
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
        outputs = outputs[:, inputs["input_ids"].shape[1]:]
        return tokenizer.decode(outputs[0], skip_special_tokens=True)
    

In [ ]:
# 开始对话
prompt = "只切一刀，如何把四个橘子分给三个小朋友？"
response = chat(prompt)
print(response)